# Load reviewed Suite2p ROIs

The ROI reviewer starts with the preprocessing pipeline's morphology QC target structure applied. Morphology failures are initially Bad and passing ROIs are initially Unlabeled. After review, place `roi_manual_labels.npy` in the session's `suite2p/plane0/` directory.

## Generate the interactive labeler and PDF summary

Run these commands from the root of a `2p_imaging` checkout. The session path must point to an already processed session containing at least `suite2p/plane0/ops.npy`, `stat.npy`, `F.npy`, and `Fneu.npy`.

### Download and install the Conda environment

Suite2p 1.x is the recommended environment:

```bash
curl -L -o environment-preprocessing-qc-suite2p-1x.yml \
  https://raw.githubusercontent.com/najafi-laboratory/2p_imaging/main/utils_2p/environment-preprocessing-qc-suite2p-1x.yml

conda env create \
  --prefix ~/conda/envs/2p_preprocessing_qc_suite2p_1x \
  --file environment-preprocessing-qc-suite2p-1x.yml

conda activate ~/conda/envs/2p_preprocessing_qc_suite2p_1x
```

For the legacy Suite2p 0.x environment:

```bash
curl -L -o environment-preprocessing-qc-suite2p-0x.yml \
  https://raw.githubusercontent.com/najafi-laboratory/2p_imaging/main/utils_2p/environment-preprocessing-qc-suite2p-0x.yml

conda env create \
  --prefix ~/conda/envs/2p_preprocessing_qc_suite2p_0x \
  --file environment-preprocessing-qc-suite2p-0x.yml
```

### Generate locally

```bash
python -m utils_2p.preprocessing_summary /path/to/processed/session
```

This writes `<session>_interactive_fov_roi_dff.html` and `<session>_preprocessing_summary.pdf` into the processed session directory.

### Generate on PACE

The shared Suite2p 1.x environment can be used without creating a personal environment. Submit the summary on a compute node rather than running it on the login node:

```bash
export TWO_P_PYTHON=/storage/project/r-fnajafi3-0/grubin6/shared_envs/2p_preprocessing_qc_suite2p_1x/bin/python

sbatch \
  --account=gts-fnajafi3 \
  --qos=embers \
  --cpus-per-task=4 \
  --mem=24G \
  --time=02:00:00 \
  --job-name=preprocessing_summary \
  --wrap="$TWO_P_PYTHON -m utils_2p.preprocessing_summary /path/to/processed/session"
```

To install a personal environment on PACE instead, first run `module load anaconda3/2023.03`, then use the same `curl` and `conda env create` commands above. Set `TWO_P_PYTHON` to the resulting environment's `bin/python`.

In [ ]:
from utils_2p.roi_labels import load_reviewed_dff

In [ ]:
session = load_reviewed_dff("/path/to/session")

dff = session["dff"]
roi_indices = session["roi_indices"]

print(dff.shape)
print(roi_indices)

`dff` contains the morphology-filtered Good ROI traces selected by `roi_manual_labels.npy` and has shape `(selected_rois, frames)`. `roi_indices` identifies the corresponding rows in the original Suite2p files. To include Unsure ROIs with Good ROIs, call `load_reviewed_dff("/path/to/session", policy="good_or_unsure")`. To use a `roi_manual_labels.npy` stored elsewhere, pass its path as `label_path`.